In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from numpy import mean
from numpy import std
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

/usr/local/lib/python3.12/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:19: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [5]:
!git clone https://github.com/A2AppRom/sophia-romberg-data

Cloning into 'sophia-romberg-data'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 79 (delta 9), reused 72 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 4.60 MiB | 12.90 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [6]:
# define paths to train, test split
DATA_ROOT = "/content/sophia-romberg-data/labeled_data_with_features_NEW.csv"

In [7]:
df = pd.read_csv(DATA_ROOT)

In [8]:
df = df.drop(columns="Unnamed: 0")

In [ ]:
import random

n = random.choices([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], k=5)
print(n)

for i in range(10):
  if i in n:
    df.at[i, "label"] = "impaired"

[0, 6, 2, 9, 7]


In [9]:
df

,subject_id,session_id,label,duration,rms,std_mag,romberg_ratio_rms,romberg_ratio_std
0,1,0,open,58753421600,0.584403,0.556785,1.101173,1.062279
1,1,0,closed,45567372600,0.643529,0.591461,1.101173,1.062279
2,2,0,open,42846806800,0.366372,0.328285,1.124497,1.145674
3,2,0,closed,43068919300,0.411984,0.376108,1.124497,1.145674
4,3,0,open,42798669300,0.383673,0.350194,1.040417,1.029774
5,3,0,closed,42738772700,0.399179,0.360620,1.040417,1.029774
6,4,0,open,32043969000,0.736683,0.621835,0.510672,0.496044
7,4,0,closed,34405365500,0.376203,0.308457,0.510672,0.496044
8,0,0,open,45564259500,0.400963,0.342887,1.479420,1.619363
9,0,0,closed,45324047600,0.593192,0.555259,1.479420,1.619363


In [10]:
train, test= train_test_split(df, test_size=0.2)

In [11]:
# determine need for class weighting
targets = ["label"]

for t in targets:
    print(t)
    print(train[t].value_counts(normalize=True))
    print()

label
label
closed    0.53125
open      0.46875
Name: proportion, dtype: float64



In [12]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

Train shape: (32, 8)
Test shape: (8, 8)


,subject_id,session_id,label,duration,rms,std_mag,romberg_ratio_rms,romberg_ratio_std
0,1,0,open,58753421600,0.584403,0.556785,1.101173,1.062279
19,7,0,closed,43120434200,0.638751,0.600707,0.631156,0.614811
31,8,5,closed,30781666300,0.078172,0.045329,1.319377,1.595739
24,8,2,open,30781370800,0.059968,0.029631,1.057429,1.036647
7,4,0,closed,34405365500,0.376203,0.308457,0.510672,0.496044


In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32 entries, 0 to 33
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   subject_id         32 non-null     int64  
 1   session_id         32 non-null     int64  
 2   label              32 non-null     object 
 3   duration           32 non-null     int64  
 4   rms                32 non-null     float64
 5   std_mag            32 non-null     float64
 6   romberg_ratio_rms  32 non-null     float64
 7   romberg_ratio_std  32 non-null     float64
dtypes: float64(4), int64(3), object(1)
memory usage: 2.2+ KB


In [14]:
numeric_cols = train.select_dtypes(include=np.number).columns
categorical_cols = train.select_dtypes(exclude=np.number).columns

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

Numeric: 7
Categorical: 1


In [15]:
targets = ["label"]
X = train.drop(columns=targets)
y = train["label"]  # start with one horizon

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [16]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]

In [17]:
print("ROC AUC:", roc_auc_score(y_val, val_probs))

ROC AUC: 0.25


In [18]:
y_val

,label
35,closed
3,closed
27,closed
24,open
22,open
23,closed
18,open


In [19]:
val_probs

array([0.45322134, 0.61534768, 0.43499289, 0.39494569, 0.84067047,
       0.84363383, 0.40079106])

HistGradientBoostingClassifier + Generate Synthetic Data

In [ ]:
X, y = make_classification(n_samples=100, n_features=11, n_informative=3, n_redundant=8, random_state=1)
model = HistGradientBoostingClassifier(max_bins=255, max_iter=100)
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
# evaluate the model and collect the scores
n_scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=-1)
print('Accuracy: %.3f (%.3f)' % (mean(n_scores), std(n_scores)))

Accuracy: 0.923 (0.084)


XGBoost

In [ ]:
!sudo pip install xgboost

In [ ]:
# define dataset for example cross-validation only
X_synthetic, y_synthetic = make_classification(n_samples=100, n_features=11, n_informative=3, n_redundant=8, random_state=1)
# define the model for this example
example_model = XGBClassifier(tree_method='approx', max_bin=255, n_estimators=1000)
# define the evaluation procedure
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
# evaluate the model and collect the scores
n_scores = cross_val_score(example_model, X_synthetic, y_synthetic, scoring='accuracy', cv=cv, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (mean(n_scores), std(n_scores)))

Accuracy: 0.920 (0.101)
